# 秋田県クマ出没予測 — IBM Granite TTM（全グリッド 260 セル版）

**入力 CSV**: `Akita_10km_AllGrid_260cells_Daily_TimeSeries.csv`  
　　　　　　（`generate_akita_allgrid_colab.ipynb` で生成）  
**モデル**: `ibm/granite-ttm-512-96-r2`（コンテキスト 512 日, 予測上限 96 日）  
**グリッド**: 13col × 20row = **260 セル**（出没なし → 0 埋め済み）  
**予測期間**: 2025-01-01 〜 2025-12-31（4 ウィンドウ × 96 日）  

| 評価 | 方式 | 用途 |
|------|------|------|
| セルごと | PR-AUC / ROC-AUC / Brier / ECE / MAE / RMSE | セル単体の予測品質 |
| **グローバル** | **P@K / R@K (K=10,20,30)** — 全 260 セル日次ランキング | **ET との直接比較** |

> ET (`benchmark_akita_10km_v2.py`) も 260 セルのグローバル評価のため直接比較可能  
> ランダム基準: P_rnd = avg_daily_pos / 260

## 0) 依存関係のインストール

In [ ]:
!pip install -q japanize_matplotlib

In [ ]:
!pip install -q \
  "numpy==2.0.2" \
  "pandas==2.2.2" \
  "scikit-learn==1.4.2" \
  "matplotlib>=3.8,<3.9" \
  "seaborn>=0.13,<0.14" \
  "ibm-watsonx-ai>=1.1.9,<2" \
  --upgrade --upgrade-strategy only-if-needed

In [ ]:
import numpy, pandas, sklearn, ibm_watsonx_ai, matplotlib, seaborn
for mod in [numpy, pandas, sklearn, ibm_watsonx_ai, matplotlib, seaborn]:
    print(f'{mod.__name__:<20}: {mod.__version__}')

In [ ]:
!pip install -q git+https://github.com/ibm-granite-community/utils

## 1) IBM watsonx.ai 認証

In [ ]:
import os

os.environ['WX_APIKEY']     = 'REDACTED_IBM_APIKEY'
os.environ['WX_URL']        = 'https://jp-tok.ml.cloud.ibm.com'
os.environ['WX_PROJECT_ID'] = 'REDACTED_IBM_PROJECT_ID'

assert all(os.environ.get(k) for k in ['WX_APIKEY','WX_URL','WX_PROJECT_ID'])
print('認証設定 OK')

## 2) データ読み込み

`Akita_10km_AllGrid_260cells_Daily_TimeSeries.csv` をアップロードしてください。

In [ ]:
import numpy as np
import pandas as pd
from google.colab import files

print('Akita_10km_AllGrid_260cells_Daily_TimeSeries.csv をアップロードしてください')
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
print(f'読み込みファイル: {CSV_PATH}')

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])

# メタ列以外を LOCATION_COLS として動的検出
META_COLS     = {'Date', 'Year', 'Month', 'Week', 'Weekday', 'Sum'}
LOCATION_COLS = [c for c in df_raw.columns if c not in META_COLS]
N_CELLS       = len(LOCATION_COLS)

for col in LOCATION_COLS + ['Sum']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce').fillna(0).astype(int)

print(f'元データ   : {len(df_raw):,} 行')
print(f'期間       : {df_raw["Date"].min().date()} ~ {df_raw["Date"].max().date()}')
print(f'セル数     : {N_CELLS}')
print(f'Sum 最大値 : {df_raw["Sum"].max()}  (= 1日の出没セル数 最大)')

# 訓練・テスト別統計
for label, mask in [
    ('訓練 (2020-2024)', df_raw['Year'] < 2025),
    ('テスト (2025)   ', df_raw['Year'] == 2025),
]:
    sub = df_raw.loc[mask]
    dp  = sub[LOCATION_COLS].sum(axis=1)
    print(f'{label}: {len(sub):,}日, '
          f'出没あり {(dp>0).sum()}日 ({100*(dp>0).mean():.1f}%), '
          f'avg {dp.mean():.2f} cells/day')

In [ ]:
# 訓練期間（2020-2024）で出没ゼロのセルを特定
train_mask   = df_raw['Year'] < 2025
ZERO_CELLS   = [c for c in LOCATION_COLS
                if df_raw.loc[train_mask, c].sum() == 0]
ACTIVE_CELLS = [c for c in LOCATION_COLS if c not in ZERO_CELLS]

print(f'訓練期間アクティブセル : {len(ACTIVE_CELLS)} / {N_CELLS}')
print(f'訓練期間ゼロセル       : {len(ZERO_CELLS)} / {N_CELLS}')
print(f'  → ゼロセルは API 呼び出しをスキップし pred=0.0 を割り当て')

In [ ]:
# 連続日次データ（0 補完）
all_days = pd.date_range(df_raw['Date'].min(), df_raw['Date'].max(), freq='D')
df_full  = pd.DataFrame({'Date': all_days})
df_full  = df_full.merge(
    df_raw[['Date'] + LOCATION_COLS + ['Sum']], on='Date', how='left'
)
df_full[LOCATION_COLS + ['Sum']] = (
    df_full[LOCATION_COLS + ['Sum']].fillna(0).astype(int)
)
df_full['timestamp'] = df_full['Date']

# ロング形式
df_long = df_full.melt(
    id_vars=['timestamp'], value_vars=LOCATION_COLS,
    var_name='GridID', value_name='count',
).sort_values(['GridID','timestamp']).reset_index(drop=True)
df_long['count'] = df_long['count'].astype(float)

print(f'ロング形式: {len(df_long):,} 行  ({N_CELLS} cells x {len(all_days)} days)')

## 3) データの可視化

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib

# 日次合計
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(df_full['Date'], df_full['Sum'], lw=0.7, color='steelblue')
ax.axvline(pd.Timestamp('2025-01-01'), color='red', ls='--', alpha=0.6, label='テスト開始')
ax.set_title(f'秋田県 クマ出没 日次合計（全 {N_CELLS} セル / アクティブ {len(ACTIVE_CELLS)} セル）')
ax.set_xlabel('日付'); ax.set_ylabel('出没セル数')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# 出没日数 TOP5 セル
train_mask = df_full['Date'] < pd.Timestamp('2025-01-01')
top5 = (
    df_full.loc[train_mask, LOCATION_COLS]
    .apply(lambda c: (c > 0).sum())
    .sort_values(ascending=False)
    .head(5).index.tolist()
)
print(f'出没日数 TOP5: {top5}')

fig, axes = plt.subplots(5, 1, figsize=(16, 10), sharex=True)
for ax, gid in zip(axes, top5):
    ax.bar(df_full['Date'], df_full[gid], width=1, alpha=0.7)
    ax.axvline(pd.Timestamp('2025-01-01'), color='red', ls='--', alpha=0.4)
    ax.set_ylabel(gid, fontsize=9); ax.set_ylim(-0.05, 1.3)
    ax.grid(True, alpha=0.2)
axes[0].set_title('出没日数 TOP5 セル')
axes[-1].set_xlabel('日付')
plt.tight_layout(); plt.show()

## 4) Watsonx Granite TTM 初期化

In [ ]:
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.foundation_models import TSModelInference

creds    = Credentials(url=os.environ['WX_URL'], api_key=os.environ['WX_APIKEY'])
client   = APIClient(creds)
client.set.default_project(os.environ['WX_PROJECT_ID'])

TTM_MODEL_ID = 'ibm/granite-ttm-512-96-r2'
ts_model = TSModelInference(
    model_id=TTM_MODEL_ID,
    credentials=creds,
    project_id=os.environ['WX_PROJECT_ID'],
)
print(f'Model             : {TTM_MODEL_ID}')
print(f'Context length    : 512 days')
print(f'Active cells      : {len(ACTIVE_CELLS)} (API あり)')
print(f'Zero-train cells  : {len(ZERO_CELLS)} (pred=0.0 固定, API なし)')
print(f'API calls (est.)  : {len(ACTIVE_CELLS)} x 4 = {len(ACTIVE_CELLS)*4} req')
print(f'Time (est.)       : ~{len(ACTIVE_CELLS)*4*1.5/60:.0f} min')

## 5) 全セル ローリング予測（2025 年）

- **アクティブセル** (`{len(ACTIVE_CELLS)}` 個): TTM API でローリング予測  
- **ゼロセル** (`{len(ZERO_CELLS)}` 個): 訓練期間に出没なし → `pred = 0.0` を直接代入（API 呼び出し不要）

ゼロセルは TTM に入力しても予測値 ≈ 0 になるため、グローバル P@K/R@K の結果への影響は最小。

In [ ]:
import time

TRAIN_START    = pd.Timestamp('2020-04-01')
TRAIN_END      = pd.Timestamp('2024-12-31')
FORECAST_START = pd.Timestamp('2025-01-01')
FORECAST_END   = pd.Timestamp('2025-12-31')

ROLLING_STEP       = 96
CONTEXT_LEN        = 512
INTER_REQUEST_WAIT = 1.0
MAX_RETRIES        = 6
RETRY_BASE_WAIT    = 5.0

def forecast_with_retry(ts_model, payload, params, grid_id, window_start):
    wait = RETRY_BASE_WAIT
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return ts_model.forecast(data=payload, params=params)
        except Exception as exc:
            msg = str(exc)
            if ('429' in msg or 'rate_limit' in msg.lower()) and attempt < MAX_RETRIES:
                print(f'  RateLimit {grid_id} [{window_start.date()}]: '
                      f'wait {wait:.0f}s ({attempt}/{MAX_RETRIES})')
                time.sleep(wait)
                wait = min(wait * 2, 120)
            else:
                raise

target_col  = 'count'
all_results = []
error_cells = []

test_days = pd.date_range(FORECAST_START, FORECAST_END, freq='D')

# ── ゼロセル: pred=0.0 を直接代入 ──────────────────────────────────────
for grid_id in ZERO_CELLS:
    zero_df = pd.DataFrame({
        'timestamp': test_days,
        'pred':      0.0,
        'GridID':    grid_id,
    })
    all_results.append(zero_df)
print(f'ゼロセル {len(ZERO_CELLS)} 個: pred=0.0 を代入完了')

# ── アクティブセル: TTM API ───────────────────────────────────────────
grid_ids = sorted(ACTIVE_CELLS)
print(f'\nアクティブセル {len(grid_ids)} 個: TTM 予測開始...')

for i, grid_id in enumerate(grid_ids, 1):
    group = (
        df_long[df_long['GridID'] == grid_id]
        .sort_values('timestamp')
        .reset_index(drop=True)
        .copy()
    )
    group['timestamp'] = pd.to_datetime(group['timestamp'])

    train_full = group[
        (group['timestamp'] >= TRAIN_START) &
        (group['timestamp'] <= TRAIN_END)
    ].copy()

    accumulated_preds = []
    grid_results      = []

    window_start = FORECAST_START
    while window_start <= FORECAST_END:
        window_end = min(
            window_start + pd.Timedelta(days=ROLLING_STEP - 1),
            FORECAST_END,
        )
        pred_len = (window_end - window_start).days + 1

        ctx_end   = window_start - pd.Timedelta(days=1)
        ctx_start = ctx_end     - pd.Timedelta(days=CONTEXT_LEN - 1)

        acc_df = (
            pd.concat(accumulated_preds, ignore_index=True)
            if accumulated_preds
            else pd.DataFrame(columns=['timestamp', target_col])
        )
        historical = (
            pd.concat(
                [train_full[train_full['timestamp'] >= ctx_start], acc_df],
                ignore_index=True,
            )
            .sort_values('timestamp')
            .drop_duplicates('timestamp')
            .reset_index(drop=True)
        )
        ctx_df = historical.tail(CONTEXT_LEN)[['timestamp', target_col]].copy()
        ctx_df[target_col] = ctx_df[target_col].astype(float)

        if len(ctx_df) < 10:
            window_start = window_end + pd.Timedelta(days=1)
            continue

        payload             = ctx_df.copy()
        payload['timestamp'] = payload['timestamp'].dt.strftime('%Y-%m-%d')
        params = {
            'timestamp_column': 'timestamp',
            'target_columns':   [target_col],
            'prediction_length': int(pred_len),
            'freq':             '1D',
        }

        try:
            time.sleep(INTER_REQUEST_WAIT)
            resp = forecast_with_retry(ts_model, payload, params, grid_id, window_start)
            obj  = resp if isinstance(resp, dict) else (
                getattr(resp, 'result', None) or resp.__dict__
            )
            node    = obj['results'][0]
            pred_df = pd.DataFrame({
                'timestamp': pd.to_datetime(node['timestamp']),
                'pred':      np.array(node[target_col]).clip(0),
                'GridID':    grid_id,
            })
            acc_rows             = pred_df[['timestamp']].copy()
            acc_rows[target_col] = pred_df['pred'].values
            accumulated_preds.append(acc_rows)
            grid_results.append(pred_df)

        except Exception as exc:
            print(f'  ERROR {grid_id} [{window_start.date()}]: {exc}')
            error_cells.append(grid_id)

        window_start = window_end + pd.Timedelta(days=1)

    if grid_results:
        all_results.extend(grid_results)
        if i % 20 == 0 or i == len(grid_ids):
            print(f'  [{i:3d}/{len(grid_ids)}] done through {grid_id}')

print(f'\n予測完了 / エラーセル: {list(set(error_cells)) if error_cells else "なし"}')

## 6) 予測結果の統合と実測値の照合

In [ ]:
df_pred = pd.concat(all_results, ignore_index=True)
df_pred['timestamp'] = pd.to_datetime(df_pred['timestamp'])

df_actual = (
    df_long[
        df_long['timestamp'].between(FORECAST_START, FORECAST_END)
    ]
    .rename(columns={'count': 'actual'})
)

df_all_pred = df_pred.merge(
    df_actual[['GridID','timestamp','actual']],
    on=['GridID','timestamp'], how='left',
)
df_all_pred['abs_error'] = (df_all_pred['pred'] - df_all_pred['actual']).abs()

n_pred  = df_all_pred['GridID'].nunique()
n_total = N_CELLS
print(f'予測セル数: {n_pred} / {n_total}')
assert n_pred == n_total, f'セル数不一致: {n_pred} != {n_total}'

eval_df = df_all_pred.groupby('GridID').agg(
    MAE=('abs_error','mean'), N=('pred','count')
).reset_index()
print(f'MAE: mean={eval_df["MAE"].mean():.3f} '
      f'min={eval_df["MAE"].min():.3f} '
      f'max={eval_df["MAE"].max():.3f}')

## 7) セルごと評価

ROC-AUC / PR-AUC / Brier / ECE / MAE / RMSE および  
**セル時間軸 P@K**: 「このセルについて上位 K% の日を予測できたか」（TOP20 版との互換性維持）

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

def ece_score(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        m = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if m.sum() == 0:
            continue
        ece += (m.sum()/len(y_true)) * abs(y_prob[m].mean() - y_true[m].mean())
    return ece

def pk_rk_pct(g, y_bin, k_pct):
    """Per-cell temporal P@K: top k% of days for this cell."""
    k  = max(int(len(g) * k_pct / 100), 1)
    ti = g.sort_values('pred', ascending=False).head(k).index
    tp = len(set(ti) & set(g[y_bin == 1].index))
    return tp / k, tp / max(y_bin.sum(), 1)

df_eval     = df_all_pred.dropna(subset=['actual']).copy()
grid_scores = []

for grid_id, g in df_eval.groupby('GridID'):
    y_true = g['actual'].fillna(0).astype(float).values
    y_pred = g['pred'].astype(float).values
    y_bin  = (y_true > 0).astype(int)

    if y_bin.sum() > 0 and y_bin.sum() < len(y_bin):
        try:
            pr_auc  = average_precision_score(y_bin, y_pred)
            roc_auc = roc_auc_score(y_bin, y_pred)
            brier   = brier_score_loss(y_bin, y_pred.clip(0,1))
        except Exception:
            pr_auc = roc_auc = brier = np.nan
    else:
        pr_auc = roc_auc = brier = np.nan

    ece  = ece_score(y_bin, y_pred.clip(0,1))
    mae  = np.mean(np.abs(y_pred - y_true))
    rmse = np.sqrt(np.mean((y_pred - y_true)**2))

    topk_n       = max(int(len(g)*0.10), 1)
    pred_top_idx = g.sort_values('pred', ascending=False).head(topk_n).index
    hit_top10    = len(set(pred_top_idx) & set(g[y_bin==1].index)) / max(y_bin.sum(),1)

    p10,r10 = pk_rk_pct(g, y_bin, 10)
    p20,r20 = pk_rk_pct(g, y_bin, 20)
    p30,r30 = pk_rk_pct(g, y_bin, 30)

    grid_scores.append(dict(
        grid_id=grid_id, PR_AUC=pr_auc, ROC_AUC=roc_auc,
        Brier=brier, ECE=ece, MAE=mae, RMSE=rmse,
        Hit_Top10=hit_top10,
        Precision_10=p10, Precision_20=p20, Precision_30=p30,
        Recall_10=r10,    Recall_20=r20,    Recall_30=r30,
    ))

score_df = pd.DataFrame(grid_scores)
valid_df = score_df.dropna(subset=['ROC_AUC'])

print(f'評価可能セル: {len(valid_df)} / {len(score_df)}')
print('\n--- 評価可能セル 平均 ---')
print(valid_df[[
    'PR_AUC','ROC_AUC','Brier','ECE','MAE','RMSE',
    'Precision_10','Recall_10','Precision_20','Recall_20','Precision_30','Recall_30'
]].mean().round(4).to_string())

## 8) グローバル P@K / R@K 評価（全 260 セル日次ランキング）

ET の `benchmark_akita_10km_v2.py` と**同一の式**で評価します。

```
P@K(day) = hits / K          hits = top-K に入った実際の出没セル数
R@K(day) = hits / n_pos      n_pos = 当日の全出没セル数（260 セル中）
```

ランダム基準: `P_rnd = avg_daily_pos / 260`  
ET グローバル結果 (参考): P@10=0.168 (2.7x), P@20=0.178 (2.9x), P@30=0.182 (3.0x)

In [ ]:
K_VALUES = [10, 20, 30]

# ワイド形式: timestamp x GridID (全 260 セル)
pred_wide = (
    df_pred
    .pivot(index='timestamp', columns='GridID', values='pred')
    .reindex(columns=LOCATION_COLS, fill_value=0)
    .fillna(0)
)

actual_wide = (
    df_long[df_long['timestamp'].between(FORECAST_START, FORECAST_END)]
    .pivot(index='timestamp', columns='GridID', values='count')
    .reindex(columns=LOCATION_COLS, fill_value=0)
    .fillna(0)
)

assert pred_wide.shape[1]   == N_CELLS, f'pred cells {pred_wide.shape[1]} != {N_CELLS}'
assert actual_wide.shape[1] == N_CELLS, f'actual cells {actual_wide.shape[1]} != {N_CELLS}'

# 日次ランキング
day_p = {k: [] for k in K_VALUES}
day_r = {k: [] for k in K_VALUES}

for ts in pred_wide.index:
    if ts not in actual_wide.index:
        continue
    ranked  = pred_wide.loc[ts].sort_values(ascending=False).index.tolist()
    present = set(actual_wide.loc[ts][actual_wide.loc[ts] > 0].index)
    n_pos   = len(present)
    for k in K_VALUES:
        hits = len(set(ranked[:k]) & present)
        day_p[k].append(hits / k)
        if n_pos > 0:
            day_r[k].append(hits / n_pos)

avg_daily_pos = actual_wide.gt(0).sum(axis=1).mean()
rnd_p         = avg_daily_pos / N_CELLS

print(f'セルプール数   : {N_CELLS}')
print(f'avg_daily_pos  : {avg_daily_pos:.2f} cells/day')
print(f'ランダム P@K   : {rnd_p:.4f} ({avg_daily_pos:.1f}/{N_CELLS})')
print()
print(f'{"":4} {"P@K":>8} {"vs rnd":>8} {"R@K":>8}')
print('-' * 32)

global_rows = []
for k in K_VALUES:
    p_mean = float(np.nanmean(day_p[k]))
    r_mean = float(np.nanmean(day_r[k]))
    mult   = p_mean / rnd_p if rnd_p > 0 else float('nan')
    print(f'K={k:2d}  {p_mean:8.4f}  {mult:6.1f}x  {r_mean:8.4f}')
    global_rows.append({
        'K': k, 'P@K': p_mean, 'R@K': r_mean,
        'P_mult_vs_random': mult,
        'n_cells_pool': N_CELLS,
        'avg_daily_pos': round(avg_daily_pos, 3),
        'rnd_P': round(rnd_p, 5),
    })

global_pk_df = pd.DataFrame(global_rows)

## 9) ET との比較表（グローバル P@K/R@K）

In [ ]:
# ET 結果（benchmark_akita_10km_v2.py より, 260 cells）
ET_RESULTS = {
    10: {'P@K': 0.168, 'R@K': 0.106, 'mult': 2.7},
    20: {'P@K': 0.178, 'R@K': 0.215, 'mult': 2.9},
    30: {'P@K': 0.182, 'R@K': 0.346, 'mult': 3.0},
}

print(f'Akita Global P@K/R@K 比較（全 {N_CELLS} セル, ランダム基準={rnd_p:.4f}）')
print()
print(f'{"":4} {"TTM-512 P@K":>12} {"TTM-512 R@K":>12} {"ET P@K":>10} {"ET R@K":>10}')
print('-' * 52)
for k in K_VALUES:
    row = global_pk_df[global_pk_df['K'] == k].iloc[0]
    et  = ET_RESULTS[k]
    winner_p = 'TTM' if row['P@K'] >= et['P@K'] else 'ET '
    winner_r = 'TTM' if row['R@K'] >= et['R@K'] else 'ET '
    print(f'K={k:2d}  '
          f'{row["P@K"]:10.4f} ({row["P_mult_vs_random"]:.1f}x)  '
          f'{row["R@K"]:10.4f}    '
          f'{et["P@K"]:8.4f} ({et["mult"]:.1f}x)  '
          f'{et["R@K"]:8.4f}  '
          f'P->{winner_p} R->{winner_r}')

## 10) 評価指標の可視化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib

# グローバル P@K/R@K 棒グラフ
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
k_labels = [f'P@{k}' for k in K_VALUES]
p_vals   = global_pk_df['P@K'].tolist()
r_vals   = global_pk_df['R@K'].tolist()
et_p     = [ET_RESULTS[k]['P@K'] for k in K_VALUES]
et_r     = [ET_RESULTS[k]['R@K'] for k in K_VALUES]

x = np.arange(len(K_VALUES))
w = 0.35
axes[0].bar(x - w/2, p_vals, w, label='TTM-512', color='steelblue')
axes[0].bar(x + w/2, et_p,   w, label='ET',      color='darkorange', alpha=0.8)
axes[0].axhline(rnd_p, color='red', ls='--', label=f'random ({rnd_p:.3f})')
axes[0].set_xticks(x); axes[0].set_xticklabels(k_labels)
axes[0].set_title('Global Precision@K (260 cells)'); axes[0].legend(fontsize=8)

axes[1].bar(x - w/2, r_vals, w, label='TTM-512', color='steelblue')
axes[1].bar(x + w/2, et_r,   w, label='ET',      color='darkorange', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels([f'R@{k}' for k in K_VALUES])
axes[1].set_title('Global Recall@K (260 cells)'); axes[1].legend(fontsize=8)

plt.suptitle(f'秋田県 TTM-512 vs ET — グローバル P@K/R@K ({N_CELLS} セル)', fontsize=12)
plt.tight_layout(); plt.show()

# セルごと ROC-AUC ヒストグラム
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(valid_df['ROC_AUC'].dropna(), bins=20, color='steelblue', edgecolor='white')
ax.axvline(0.5, color='red', ls='--', label='random (0.5)')
ax.axvline(valid_df['ROC_AUC'].mean(), color='green', ls='--',
           label=f'mean ({valid_df["ROC_AUC"].mean():.3f})')
ax.set_title(f'セルごと ROC-AUC 分布 ({len(valid_df)} セル)')
ax.set_xlabel('ROC-AUC'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 11) CSV に保存

In [ ]:
score_df.to_csv('akita_260cells_percell_evaluation_2025.csv', index=False)
print(f'セルごと評価  : akita_260cells_percell_evaluation_2025.csv  ({len(score_df)} rows)')

df_all_pred.to_csv('akita_260cells_predictions_2025.csv', index=False)
print(f'全予測結果    : akita_260cells_predictions_2025.csv')

global_pk_df.to_csv('akita_260cells_global_pkrk_2025.csv', index=False)
print(f'グローバル P@K: akita_260cells_global_pkrk_2025.csv')

## 12) 上位・下位グリッドの確認

In [ ]:
cols_show = ['grid_id','ROC_AUC','PR_AUC','Brier','ECE','MAE','Recall_20']

print('ROC-AUC 上位 10 セル:')
print(valid_df.sort_values('ROC_AUC', ascending=False).head(10)[cols_show].to_string(index=False))

print('\nROC-AUC 下位 10 セル:')
print(valid_df.sort_values('ROC_AUC').head(10)[cols_show].to_string(index=False))

## 13) 予測 vs 実測の可視化

In [ ]:
import japanize_matplotlib

best_cell = valid_df.sort_values('ROC_AUC', ascending=False).iloc[0]['grid_id']
plot_df   = df_all_pred[df_all_pred['GridID'] == best_cell]
roc_val   = valid_df.loc[valid_df['grid_id']==best_cell,'ROC_AUC'].values[0]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(plot_df['timestamp'], plot_df['actual'], label='実測', marker='o', ms=3, lw=0.8)
ax.plot(plot_df['timestamp'], plot_df['pred'],   label='予測', marker='x', ms=3, lw=0.8)
ax.set_title(f'Cell {best_cell} — 実測 vs 予測 (2025, ROC-AUC={roc_val:.3f})')
ax.set_xlabel('日付'); ax.set_ylabel('出没'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
import os

# 2025 年に実際の出没があったセルのみ保存（プロット数を抑制）
test_active = [
    gid for gid in df_all_pred['GridID'].unique()
    if df_all_pred.loc[df_all_pred['GridID']==gid,'actual'].sum() > 0
]
print(f'2025 年出没セル: {len(test_active)} / {df_all_pred["GridID"].nunique()}')

output_dir = 'plots_akita_260cells_2025'
os.makedirs(output_dir, exist_ok=True)

for grid_id in test_active:
    g   = df_all_pred[df_all_pred['GridID'] == grid_id]
    rv  = score_df.loc[score_df['grid_id']==grid_id,'ROC_AUC'].values
    roc = f', ROC={rv[0]:.3f}' if len(rv)>0 and not np.isnan(rv[0]) else ''
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(g['timestamp'], g['actual'], label='実測', ms=2, lw=0.7, marker='o')
    ax.plot(g['timestamp'], g['pred'],   label='予測', ms=2, lw=0.7, marker='x')
    ax.set_title(f'Cell {grid_id} (2025{roc})')
    ax.set_xlabel('日付'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'akita_{grid_id}.png'), dpi=120)
    plt.close()

print(f'{len(test_active)} 枚を {output_dir}/ に保存')

## 14) ROC 曲線 / PR 曲線（上位 5 セル）

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

top5_grids = valid_df.sort_values('ROC_AUC', ascending=False).head(5)['grid_id'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for gid in top5_grids:
    g      = df_eval[df_eval['GridID'] == gid]
    y_true = (g['actual'].fillna(0) > 0).astype(int)
    y_pred = g['pred'].astype(float)
    if y_true.sum() == 0 or y_true.sum() == len(y_true):
        continue
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    axes[0].plot(fpr, tpr, label=f'{gid} ({auc(fpr,tpr):.3f})')
    prec, rec, _ = precision_recall_curve(y_true, y_pred)
    axes[1].plot(rec, prec, label=f'{gid} ({auc(rec,prec):.3f})')

axes[0].plot([0,1],[0,1],'k--',alpha=0.3); axes[0].set_title('ROC 曲線（上位 5）')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title('PR 曲線（上位 5）')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.suptitle(f'秋田県 TTM-512 全グリッド版 ({N_CELLS} セル) — 2025 年', fontsize=12)
plt.tight_layout(); plt.show()

## 15) ダウンロード

In [ ]:
from google.colab import files

files.download('akita_260cells_percell_evaluation_2025.csv')
files.download('akita_260cells_predictions_2025.csv')
files.download('akita_260cells_global_pkrk_2025.csv')
print('ダウンロード完了')